### Q2 & Q3

In [2]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import math

# small dataset being used for this part
sentences = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "birds fly in the sky",
    "the child plays with toys",
    "the sun shines brightly",
    "the moon glows at night",
    "flowers bloom in spring",
    "rain falls from clouds",
    "the fish swims in water",
    "the baby sleeps quietly"
]

In [6]:
tokens = [s.split() for s in sentences]

vocab = sorted(set(word for sent in tokens for word in sent))
stoi = {w:i+1 for i,w in enumerate(vocab)}
itos = {i:w for w,i in stoi.items()}

vocab_size = len(stoi) + 1
max_len = max(len(s) for s in tokens)

# encode our tokens with numbers
encoded = []
for sent in tokens:
    row = [stoi[w] for w in sent]
    row += [0] * (max_len - len(row))
    encoded.append(row)

x = torch.tensor(encoded)

# Embed our word encodings
d_model = 16
embedding = nn.Embedding(vocab_size, d_model)

embedded = embedding(x)

def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (i / d_model))
            pe[pos, i] = math.sin(angle)
            if i + 1 < d_model:
                pe[pos, i+1] = math.cos(angle)
    
    return pe

pe = positional_encoding(max_len, d_model)

embedded = embedded + pe

#### SELF-ATTENTION LAYER

In [ ]:
class SelfAttentionHead(nn.Module):
    def __init__(self, d_model, head_dim):
        super().__init__()
        self.q = nn.Linear(d_model, head_dim)
        self.k = nn.Linear(d_model, head_dim)
        self.v = nn.Linear(d_model, head_dim)
        
    def forward(self, x):
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)
        
        # HERE IS THE Attention(Q,K,V) FUNCTION as required in question 3
        scores = Q @ K.transpose(-2, -1) / math.sqrt(K.size(-1))
        weights = torch.softmax(scores, dim=-1)
        output = weights @ V
        
        return output, weights

#### MULTI-HEAD

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        head_dim = d_model // num_heads
        self.heads = nn.ModuleList(
            [SelfAttentionHead(d_model, head_dim) for _ in range(num_heads)]
        )
        self.linear = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        outputs = []
        attention_weights = []
        
        for head in self.heads:
            out, weights = head(x)
            outputs.append(out)
            attention_weights.append(weights)
        
        concat = torch.cat(outputs, dim=-1)
        out = self.linear(concat)
        
        return out, attention_weights

#### Final encoder

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model=16, num_heads=2, ff_dim=32):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU(),
            nn.Linear(ff_dim, d_model)
        )
        
        self.norm2 = nn.LayerNorm(d_model) # HEre is the normalization
    
    # Feed forward layers
    def forward(self, x):
        attn_out, weights = self.mha(x)
        x = self.norm1(x + attn_out)
        
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        
        return x, weights

In [12]:
encoder = EncoderBlock()

output, attention_weights = encoder(embedded)

print("Final contextual embeddings shape:", output.shape)
print(output[0]) 
print("Input tokens for first sentence:")
print(tokens[0])

Final contextual embeddings shape: torch.Size([10, 6, 16])
tensor([[-0.8229, -1.3049, -0.4545,  1.9442,  0.0741, -0.3304,  1.7493, -0.1160,
         -0.5081, -0.2545, -0.2858,  0.3679, -0.9701, -1.3976,  1.5699,  0.7394],
        [ 1.4563, -0.1396, -0.1656,  0.2787, -0.5356, -0.5458,  0.0222, -2.7237,
          1.3209, -0.4941,  0.1389,  1.6407, -0.1835, -0.0700, -0.7113,  0.7114],
        [-0.0623, -1.7731,  0.0282, -0.1611, -0.9353,  0.1792, -0.4713,  1.4095,
         -1.0911,  0.3012, -0.3409,  0.9463, -1.0095, -0.4132,  1.2125,  2.1809],
        [-1.2407, -1.9830, -0.1150, -0.0589,  1.3181,  0.4131, -0.1881,  1.3011,
         -0.5793,  0.4588, -0.8507,  0.1878, -0.4878, -0.3806, -0.0714,  2.2765],
        [-1.2545, -2.4673,  0.5927,  0.9870,  0.4165, -0.2608,  1.6699,  0.1018,
         -0.2274, -0.1281, -0.0985,  0.4502, -0.6751, -1.0810,  1.2302,  0.7445],
        [-1.2781, -0.9616, -0.0428,  0.4187,  1.0300,  0.8851, -0.3696,  1.7564,
          0.2251,  0.6040, -1.2495,  0.4936, 